# Epoch-wise adaptive k-MIP: PascalVOC-SP and Peptides-func

Defaults to **linear 50 to 10** using paper-derived per-dataset training settings.

- **k changes over epochs** (paper Limitations suggestion: large → small).
- **Within an epoch, every layer uses the same k.**

**Free-tier tips**
- Runtime → Change runtime type → **T4 GPU**
- Mount Drive so disconnects don't wipe checkpoints
- Code is **cloned from GitHub** (this fork), not uploaded as a zip
- Research starts in full mode with the selected paper epoch horizon; quick/standard are independent runs.
- LapPE/RWSE stay **off**


## 0) Runtime check


In [ ]:
import torch, psutil, os
print('cuda:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))
    print('vram_gb:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 2))
print('ram_gb:', round(psutil.virtual_memory().total/1e9, 2))
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4'


## 1) Mount Drive (recommended)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/kmip_adaptive_k'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
print('DRIVE_ROOT =', DRIVE_ROOT)


## 2) Clone this GitHub repo

Uses **https://github.com/ZoyaAnwar/adaptive-k-MIP-GraphGPS** (epoch-wise adaptive-k).  
Do **not** clone the upstream paper repo — it lacks this curriculum.


In [ ]:
import os, shutil

REPO = '/content/kmip'
if os.path.exists(REPO):
    shutil.rmtree(REPO)

# Clone THIS fork (epoch-wise adaptive-k), not the upstream paper repo.
!git clone --depth 1 https://github.com/ZoyaAnwar/adaptive-k-MIP-GraphGPS.git /content/kmip
%cd /content/kmip

assert os.path.exists('main.py'), 'clone failed — main.py missing'
assert os.path.exists('graphgps/adaptive_k/train_schedule.py'), 'epoch-k module missing'
assert os.path.exists('colab/setup_env.sh'), 'colab scripts missing'
print('Repo ready at', os.getcwd())
!git rev-parse --short HEAD


### Optional: refresh after you push

Re-run the clone cell above, or:


In [ ]:
# !git -C /content/kmip pull
# %cd /content/kmip


## 3) Point datasets + results at Drive


In [ ]:
%cd /content/kmip

import os
DRIVE_ROOT = '/content/drive/MyDrive/kmip_adaptive_k'
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)

if os.path.islink('datasets') or os.path.exists('datasets'):
    if os.path.islink('datasets'):
        os.unlink('datasets')
    elif os.path.isdir('datasets') and not os.listdir('datasets'):
        os.rmdir('datasets')
if not os.path.exists('datasets'):
    os.symlink(f'{DRIVE_ROOT}/datasets', 'datasets')

print('datasets ->', os.path.realpath('datasets'))
print('cwd', os.getcwd())


## 4) Install environment


In [ ]:
!bash colab/setup_env.sh


## 5) Optional Peptides-func environment smoke test (1 epoch, tiny)

This existing test is independent of research and writes to results/smoke. Never resume it into full research training.


In [ ]:
RUN_PEPTIDES_SMOKE = False  # Manual opt-in: leave False for Run all and PascalVOC-SP runs.
if RUN_PEPTIDES_SMOKE:
    !python main.py \
      --cfg configs/LRGB/Peptides-func/colab/GPS+kmip-fixed.yaml \
      wandb.use False \
      optim.max_epoch 1 \
      train.batch_size 32 \
      model.auto_satisfy_param_budget False \
      gt.layers 2 gnn.layers_mp 2 \
      gt.dim_hidden 64 gnn.dim_inner 64 \
      gt.sparse.val_dim 16 gt.sparse.kq_dim 8 \
      out_dir /content/drive/MyDrive/kmip_adaptive_k/results/smoke \
      name_tag smoke


## 6) Select research dataset and run linear 50 to 10

| MODE | Epochs | When |
|------|--------|------|
| `quick` | 20 | independent short/debug run; selected config batch/model settings |
| `standard` | 100 | independent exploratory; selected config batch/model settings |
| `full` | config | PascalVOC-SP: 200 epochs / batch 50; Peptides-func: 250 / batch 200 |

| RUN | Meaning |
|-----|---------|
| `fixed` | constant k=15 every epoch |
| `linear` | default: k 50 to 10 over the selected paper epoch horizon |
| `exponential` | existing Peptides-func exploratory config; settings unchanged |

Resume with the same DATASET, MODE, config, SEED and OUT_ROOT; never change max_epoch. Outputs are dataset/mode/run-separated.


In [ ]:
import os
os.environ['DATASET'] = 'Peptides-func'  # PascalVOC-SP | Peptides-func
os.environ['MODE'] = 'full'             # full research | quick short/debug | standard exploratory
os.environ['RUNS'] = 'linear'           # optional fixed; exponential: Peptides-func only
os.environ['OUT_ROOT'] = '/content/drive/MyDrive/kmip_adaptive_k/results'
os.environ['SEED'] = '0'

!bash colab/run_ablation.sh


## 7) Summarize selected dataset metric (F1 / AP)


In [ ]:
import re, pathlib, json
root = pathlib.Path(os.environ['OUT_ROOT']) / os.environ['DATASET'] / os.environ['MODE']
metric = {'PascalVOC-SP': 'f1', 'Peptides-func': 'ap'}[os.environ['DATASET']]
pat = re.compile(
    rf'Best so far: epoch (\d+).*?val_{metric}: ([0-9.]+).*?test_{metric}: ([0-9.]+)'
)
rows = []
for log in sorted(root.rglob('train_*.log')):
    text = log.read_text(errors='ignore')
    matches = pat.findall(text)
    if not matches:
        continue
    epoch, vap, tap = matches[-1]
    rows.append({
        'run': log.stem.replace('train_', ''),
        'log': str(log),
        'best_epoch': int(epoch),
        f'val_{metric}': float(vap),
        f'test_{metric}': float(tap),
    })

import pandas as pd
df = pd.DataFrame(rows).sort_values(f'val_{metric}', ascending=False)
display(df)
out = root / 'summary.json'
out.write_text(json.dumps(rows, indent=2))
print('wrote', out)


## Research resume and independent exploratory runs

1. Setup, optional independent tiny smoke test, then select DATASET and start `MODE=full RUNS=linear`
2. Later sessions: same DATASET, MODE, config, SEED and OUT_ROOT (auto-resume)
3. Keep the selected paper max_epoch unchanged; never promote quick/standard runs into full
4. RUNS defaults to linear only; fixed and existing Peptides-func exponential remain available

Watch the log for lines like:
`epoch=0/249 progress=0.000 requested_k=50 on 8 layer(s) (schedule=linear)` for Peptides-func; PascalVOC-SP uses denominator 199
